In [1]:
# ══════════════════════════════════════════════════════════════════
# CELL 1: INSTALL DEPENDENCIES
# ══════════════════════════════════════════════════════════════════
!pip install -q boxmot ultralytics
print("✅ Dependencies installed.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 8.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796.9/796.9 kB 46.9 MB/s eta 0:00:00
✅ Dependencies installed.


In [2]:
# ══════════════════════════════════════════════════════════════════
# CELL 2: LOCATE DATASET & VERIFY STRUCTURE
# ══════════════════════════════════════════════════════════════════
import os
import shutil

# ── Dataset is mounted as a Kaggle input (read-only) ────────────
INPUT_ROOT = "/kaggle/input/datasets/thanhnguyenle/data-aicity-2023-track-2"

# Auto-detect DATA_ROOT: the input might have the data directly
# or inside a subfolder like 'data/' or 'train/'
if os.path.exists(os.path.join(INPUT_ROOT, "train")):
    DATA_ROOT = INPUT_ROOT
elif os.path.exists(os.path.join(INPUT_ROOT, "data", "train")):
    DATA_ROOT = os.path.join(INPUT_ROOT, "data")
else:
    # Fallback: just use INPUT_ROOT and let discovery handle it
    DATA_ROOT = INPUT_ROOT

print(f"📁 Input root:  {INPUT_ROOT}")
print(f"📁 Data root:   {DATA_ROOT}")

# Show top-level structure
print(f"\n📂 Top-level contents of input:")
for item in sorted(os.listdir(INPUT_ROOT))[:20]:
    full = os.path.join(INPUT_ROOT, item)
    if os.path.isdir(full):
        n_children = len(os.listdir(full))
        print(f"   📂 {item}/ ({n_children} items)")
    else:
        size_mb = os.path.getsize(full) / 1e6
        print(f"   📄 {item} ({size_mb:.1f} MB)")

if os.path.exists(DATA_ROOT) and DATA_ROOT != INPUT_ROOT:
    print(f"\n📂 Contents of {DATA_ROOT}:")
    for item in sorted(os.listdir(DATA_ROOT))[:20]:
        full = os.path.join(DATA_ROOT, item)
        if os.path.isdir(full):
            print(f"   📂 {item}/")
        else:
            print(f"   📄 {item}")

# Disk check — /kaggle/working is where output goes
work_usage = shutil.disk_usage("/kaggle/working")
print(f"\n💾 /kaggle/working — Free: {work_usage.free/1e9:.1f}GB / {work_usage.total/1e9:.1f}GB")

📁 Input root:  /kaggle/input/datasets/thanhnguyenle/data-aicity-2023-track-2
📁 Data root:   /kaggle/input/datasets/thanhnguyenle/data-aicity-2023-track-2

📂 Top-level contents of input:
   📄 Dataset License AIC2022.pdf (0.1 MB)
   📄 README.md (0.0 MB)
   📄 extract_vdo_frms.py (0.0 MB)
   📄 test_queries.json (0.1 MB)
   📄 test_tracks.json (2.3 MB)
   📂 train/ (3 items)
   📄 train_tracks.json (15.7 MB)
   📂 validation/ (2 items)

💾 /kaggle/working — Free: 20.9GB / 21.0GB


In [3]:
# ══════════════════════════════════════════════════════════════════
# CELL 3: DISCOVER ALL VIDEOS & LOAD MODEL + TRACKER FACTORY
# ═════════════════════════════════════════════════════════════════
import glob
import re
from pathlib import Path
from ultralytics import YOLO
from boxmot import DeepOcSort
import cv2
import numpy as np
import json
import time
from collections import defaultdict
from tqdm.auto import tqdm

# ── Discover all videos ─────────────────────────────────────────
print("🔍 Scanning for videos...")
video_registry = {}
all_videos = glob.glob(os.path.join(DATA_ROOT, "**", "vdo.avi"), recursive=True)

# If none found, also try .mp4
if not all_videos:
    all_videos = glob.glob(os.path.join(DATA_ROOT, "**", "*.mp4"), recursive=True)
    all_videos += glob.glob(os.path.join(DATA_ROOT, "**", "*.avi"), recursive=True)

for v_path in all_videos:
    match = re.search(r'(S\d+)[/\\](c\d+)', v_path)
    if match:
        scenario = match.group(1)
        camera = match.group(2)
        key = f"{scenario}_{camera}"
        roi_path = os.path.join(os.path.dirname(v_path), 'roi.jpg')
        video_registry[key] = {
            'video': v_path,
            'roi': roi_path if os.path.exists(roi_path) else None,
            'scenario': scenario,
            'camera': camera,
        }

print(f"✅ Found {len(video_registry)} videos:")
scenario_counts = defaultdict(list)
for key, info in sorted(video_registry.items()):
    scenario_counts[info['scenario']].append(key)
for sc, keys in sorted(scenario_counts.items()):
    print(f"   📂 {sc}: {len(keys)} cameras — {', '.join(sorted(keys))}")

if not video_registry:
    print("\n⚠️ No videos found! Listing all files to debug:")
    for root, dirs, files in os.walk(DATA_ROOT):
        depth = root.replace(DATA_ROOT, '').count(os.sep)
        if depth < 4:
            indent = ' ' * 2 * depth
            print(f"{indent}{os.path.basename(root)}/")
            if depth < 3:
                for f in files[:5]:
                    print(f"{indent}  {f}")
                if len(files) > 5:
                    print(f"{indent}  ... and {len(files)-5} more")

# ── Load YOLO Model ─────────────────────────────────────────────
MODEL_PATH = "/kaggle/input/models/mrkdagods/gp-yolov11-pretrained/other/default/3/best.pt"
model = YOLO(MODEL_PATH)
print(f"\n✅ YOLO model loaded: {MODEL_PATH}")

# ── Tracker Factory (fresh tracker per video) ──────────────────
def create_tracker():
    return DeepOcSort(
        reid_weights=Path("osnet_x0_25_msmt17.pt"),
        device="cuda:0",
        half=False,
        per_class=True,
        max_age=30,
        min_hits=3,
        iou_threshold=0.3,
        cmc_off=True,
        Q_xy_scaling=0.01,
        Q_s_scaling=0.0001,
    )

print("✅ Tracker factory ready (fresh instance per video).")

# ── Class Names ─────────────────────────────────────────────────
CLASS_NAMES = {
    0: "person",
    1: "vehicle",
    # Add more as needed:
    # 2: "car",
    # 3: "truck",
}

# ── Output paths ────────────────────────────────────────────────
OUTPUT_ROOT = Path("/kaggle/working")
CROPS_ROOT = OUTPUT_ROOT / "crops"
TRACKLETS_PATH = OUTPUT_ROOT / "tracklets.json"

print(f"\n📁 Output root: {OUTPUT_ROOT}")
print(f"📁 Crops root:  {CROPS_ROOT}")
print(f"📁 Tracklets:   {TRACKLETS_PATH}")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
🔍 Scanning for videos...
✅ Found 59 videos:
   📂 S01: 5 cameras — S01_c001, S01_c002, S01_c003, S01_c004, S01_c005
   📂 S02: 4 cameras — S02_c006, S02_c007, S02_c008, S02_c009
   📂 S03: 6 cameras — S03_c010, S03_c011, S03_c012, S03_c013, S03_c014, S03_c015
   📂 S04: 25 cameras — S04_c016, S04_c017, S04_c018, S04_c019, S04_c020, S04_c021, S04_c022, S04_c023, S04_c024, S04_c025, S04_c026, S04_c027, S04_c028, S04_c029, S04_c030, S04_c031, S04_c032, S04_c033, S04_c034, S04_c035, S04_c036, S04_c037, S04_c038, S04_c039, S04_c040
   📂 S05: 19 cameras — S05_c010, S05_c016, S05_c017, S05_c018, S05_c019, S05_c020, S05_c021, S05_c022, S05_c023, S05_c024, S05_c025, S05_c026, S05_c027, S05_c02

In [4]:
# ══════════════════════════════════════════════════════════════════
# CELL 4: PROCESS ALL VIDEOS — DETECTION + TRACKING
# ══════════════════════════════════════════════════════════════════

def load_roi_mask(roi_path):
    """Load ROI mask if available."""
    if roi_path and os.path.exists(roi_path):
        roi = cv2.imread(roi_path)
        if roi is not None:
            roi_gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
            _, mask = cv2.threshold(roi_gray, 127, 255, cv2.THRESH_BINARY)
            return mask
    return None

def process_single_video(vid_key, vid_info, global_id_offset=0):
    """
    Process one video: detect + track all frames.
    Returns: (tracklets_dict, stats_dict)
    Track IDs are offset by global_id_offset to ensure uniqueness across videos.
    """
    video_path = vid_info['video']
    scenario = vid_info['scenario']
    camera = vid_info['camera']
    
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"  ❌ Could not open: {video_path}")
        return {}, {'frames': 0, 'detections': 0, 'tracks': 0, 'unique_ids': 0, 'errors': 0, 'fps_video': 0, 'resolution': '0x0'}
    
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    # Load ROI mask
    roi_mask = load_roi_mask(vid_info['roi'])
    
    # Fresh tracker for this video
    tracker = create_tracker()
    
    # Storage
    tracklets = {}  # local track_id -> list of entries
    frame_idx = 0
    total_dets = 0
    total_tracks = 0
    error_count = 0
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        # Apply ROI mask if available
        if roi_mask is not None:
            frame_masked = cv2.bitwise_and(frame, frame, mask=roi_mask)
        else:
            frame_masked = frame
        
        # ── YOLO Detection ──
        results = model.predict(frame_masked, conf=0.25, iou=0.45, verbose=False)
        dets = results[0].boxes
        num_dets = len(dets)
        total_dets += num_dets
        
        if num_dets > 0:
            det_array = np.hstack([
                dets.xyxy.cpu().numpy(),
                dets.conf.cpu().numpy().reshape(-1, 1),
                dets.cls.cpu().numpy().reshape(-1, 1)
            ])
        else:
            det_array = np.empty((0, 6))
        
        # ── Tracker Update ──
        tracks = np.empty((0, 8))
        try:
            tracks = tracker.update(det_array, frame)
            total_tracks += len(tracks)
        except Exception as e:
            error_count += 1
            if error_count <= 2:
                print(f"    ⚠️ {vid_key} frame {frame_idx}: {type(e).__name__}: {e}")
        
        # ── Store Tracklets ──
        for track in tracks:
            try:
                if len(track) >= 8:
                    x1, y1, x2, y2, track_id, conf, cls, _ = track[:8]
                elif len(track) >= 7:
                    x1, y1, x2, y2, track_id, conf, cls = track[:7]
                elif len(track) >= 5:
                    x1, y1, x2, y2, track_id = track[:5]
                    conf, cls = 0.0, 0
                else:
                    continue
                
                # Apply global offset for cross-video uniqueness
                local_id = int(track_id)
                global_id = local_id + global_id_offset
                cls = int(cls)
                
                if global_id not in tracklets:
                    tracklets[global_id] = []
                
                tracklets[global_id].append({
                    "frame": frame_idx,
                    "bbox": [float(x1), float(y1), float(x2), float(y2)],
                    "conf": float(conf),
                    "class": cls,
                    "timestamp": frame_idx / fps,
                })
            except:
                pass
        
        # ── Progress (every 200 frames) ──
        if frame_idx % 200 == 0 and frame_idx > 0:
            print(f"    Frame {frame_idx:>6}/{total_frames} | "
                  f"Dets: {num_dets:>3} | "
                  f"Active: {len(tracks):>3} | "
                  f"IDs: {len(tracklets):>4}")
        
        frame_idx += 1
    
    cap.release()
    
    stats = {
        'frames': frame_idx,
        'detections': total_dets,
        'tracks': total_tracks,
        'unique_ids': len(tracklets),
        'errors': error_count,
        'fps_video': fps,
        'resolution': f"{width}x{height}",
    }
    
    return tracklets, stats


# ═══════════════════════════════════════════════════════════════
# MAIN LOOP: PROCESS ALL VIDEOS
# ═══════════════════════════════════════════════════════════════
print("=" * 70)
print("🚀 PROCESSING ALL VIDEOS")
print("=" * 70)

all_tracklets = {}          # global_track_id -> list of entries
all_tracklet_meta = {}      # global_track_id -> {vid_key, scenario, camera}
all_stats = {}              # vid_key -> stats
global_id_offset = 0
global_start = time.time()

sorted_keys = sorted(video_registry.keys())

for vid_idx, vid_key in enumerate(sorted_keys):
    vid_info = video_registry[vid_key]
    
    print(f"\n{'─'*60}")
    print(f"📹 [{vid_idx+1}/{len(sorted_keys)}] {vid_key} "
          f"({vid_info['scenario']}/{vid_info['camera']})")
    print(f"   Video: {vid_info['video']}")
    print(f"   ROI:   {'Yes' if vid_info['roi'] else 'No'}")
    print(f"   ID offset: {global_id_offset}")
    
    v_start = time.time()
    tracklets, stats = process_single_video(vid_key, vid_info, global_id_offset)
    v_elapsed = time.time() - v_start
    
    # Store tracklets with metadata
    for gid, entries in tracklets.items():
        all_tracklets[gid] = entries
        all_tracklet_meta[gid] = {
            'vid_key': vid_key,
            'scenario': vid_info['scenario'],
            'camera': vid_info['camera'],
        }
    
    # Update offset: next video's IDs start after this video's max
    if tracklets:
        max_local_id = max(tracklets.keys())
        global_id_offset = max_local_id + 1
    
    all_stats[vid_key] = stats
    speed = stats['frames'] / v_elapsed if v_elapsed > 0 else 0
    
    print(f"   ✅ Done in {v_elapsed:.1f}s ({speed:.1f} FPS) | "
          f"Frames: {stats['frames']} | "
          f"Dets: {stats['detections']} | "
          f"IDs: {stats['unique_ids']} | "
          f"Errors: {stats['errors']}")

total_elapsed = time.time() - global_start

# ═══════════════════════════════════════════════════════════════
# GLOBAL SUMMARY
# ═══════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("✅ ALL VIDEOS PROCESSED")
print("=" * 70)
print(f"  Videos processed:     {len(all_stats)}")
print(f"  Total time:           {total_elapsed:.1f}s ({total_elapsed/60:.1f}min)")
print(f"  Total frames:         {sum(s['frames'] for s in all_stats.values())}")
print(f"  Total detections:     {sum(s['detections'] for s in all_stats.values())}")
print(f"  Total unique IDs:     {len(all_tracklets)}")
print(f"  Total errors:         {sum(s['errors'] for s in all_stats.values())}")

# Per-scenario breakdown
print(f"\n📊 PER-SCENARIO BREAKDOWN:")
scenario_agg = defaultdict(lambda: {'videos': 0, 'frames': 0, 'ids': 0, 'dets': 0})
for vk, st in all_stats.items():
    sc = vk.split('_')[0]
    scenario_agg[sc]['videos'] += 1
    scenario_agg[sc]['frames'] += st['frames']
    scenario_agg[sc]['ids'] += st['unique_ids']
    scenario_agg[sc]['dets'] += st['detections']

for sc, agg in sorted(scenario_agg.items()):
    print(f"  {sc}: {agg['videos']} videos, {agg['frames']} frames, "
          f"{agg['ids']} IDs, {agg['dets']} detections")

# Per-class breakdown
class_counts = defaultdict(int)
for gid, entries in all_tracklets.items():
    cls = entries[0]['class']
    cls_name = CLASS_NAMES.get(cls, f"cls{cls}")
    class_counts[cls_name] += 1

print(f"\n📊 TRACKS BY CLASS:")
for cls_name, count in sorted(class_counts.items()):
    print(f"  {cls_name}: {count} tracks")

print("=" * 70)

SUCCESS  | DeepOcSort: det_thresh=0.3, max_age=30, max_obs=50, min_hits=3, iou_threshold=0.3, per_class=True, asso_func=iou, delta_t=3, inertia=0.2, w_association_emb=0.5, alpha_fixed_emb=0.95, aw_param=0.5, embedding_off=False, cmc_off=True, aw_off=False, Q_xy_scaling=0.01, Q_s_scaling=0.0001
INFO     | BoxMOT v16.0.10 🚀 Python-3.12.12 torch-2.9.0+cu126
CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
INFO     | osnet_x0_25_msmt17.pt
INFO     | [PID 24] Downloading ReID weights from https://drive.google.com/uc?id=1sSwXSUlj4_tHZequ_iZ8w_Jh0VaRQMqF → osnet_x0_25_msmt17.pt


🚀 PROCESSING ALL VIDEOS

────────────────────────────────────────────────────────────
📹 [1/59] S01_c001 (S01/c001)
   Video: /kaggle/input/datasets/thanhnguyenle/data-aicity-2023-track-2/train/S01/c001/vdo.avi
   ROI:   Yes
   ID offset: 0


Downloading...
From: https://drive.google.com/uc?id=1sSwXSUlj4_tHZequ_iZ8w_Jh0VaRQMqF
To: /kaggle/working/osnet_x0_25_msmt17.pt
100%|██████████| 3.06M/3.06M [00:00<00:00, 53.8MB/s]
SUCCESS  | Loaded pretrained weights from osnet_x0_25_msmt17.pt


    Frame    200/1955 | Dets:   0 | Active:   0 | IDs:   12
    Frame    400/1955 | Dets:   0 | Active:   0 | IDs:   23
    Frame    600/1955 | Dets:   2 | Active:   2 | IDs:   34
    Frame    800/1955 | Dets:   1 | Active:   1 | IDs:   44
    Frame   1000/1955 | Dets:   3 | Active:   2 | IDs:   54
    Frame   1200/1955 | Dets:   2 | Active:   2 | IDs:   61
    Frame   1400/1955 | Dets:   2 | Active:   2 | IDs:   70
    Frame   1600/1955 | Dets:   5 | Active:   4 | IDs:   79
    Frame   1800/1955 | Dets:   1 | Active:   1 | IDs:   88


SUCCESS  | DeepOcSort: det_thresh=0.3, max_age=30, max_obs=50, min_hits=3, iou_threshold=0.3, per_class=True, asso_func=iou, delta_t=3, inertia=0.2, w_association_emb=0.5, alpha_fixed_emb=0.95, aw_param=0.5, embedding_off=False, cmc_off=True, aw_off=False, Q_xy_scaling=0.01, Q_s_scaling=0.0001
INFO     | BoxMOT v16.0.10 🚀 Python-3.12.12 torch-2.9.0+cu126
CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
INFO     | osnet_x0_25_msmt17.pt
INFO     | [PID 24] Found existing ReID weights at osnet_x0_25_msmt17.pt; skipping download.
SUCCESS  | Loaded pretrained weights from osnet_x0_25_msmt17.pt


...

   ✅ Done in 304.0s (11.2 FPS) | Frames: 3407 | Dets: 24749 | IDs: 311 | Errors: 0

────────────────────────────────────────────────────────────
📹 [57/59] S05_c034 (S05/c034)
   Video: /kaggle/input/datasets/thanhnguyenle/data-aicity-2023-track-2/validation/S05/c034/vdo.avi
   ROI:   Yes
   ID offset: 4582
    Frame    200/3425 | Dets:   2 | Active:   2 | IDs:   10
    Frame    400/3425 | Dets:   5 | Active:   4 | IDs:   17
    Frame    600/3425 | Dets:   5 | Active:   5 | IDs:   32
    Frame    800/3425 | Dets:   5 | Active:   5 | IDs:   43
    Frame   1000/3425 | Dets:   7 | Active:   7 | IDs:   53
    Frame   1200/3425 | Dets:   3 | Active:   3 | IDs:   69
    Frame   1400/3425 | Dets:   6 | Active:   6 | IDs:   81
    Frame   1600/3425 | Dets:   3 | Active:   3 | IDs:   98
    Frame   1800/3425 | Dets:   5 | Active:   5 | IDs:  110
    Frame   2000/3425 | Dets:   7 | Active:   7 | IDs:  124
    Frame   2200/3425 | Dets:   4 | Active:   3 | IDs:  141
    Frame   2400/3425 | Dets:   

SUCCESS  | DeepOcSort: det_thresh=0.3, max_age=30, max_obs=50, min_hits=3, iou_threshold=0.3, per_class=True, asso_func=iou, delta_t=3, inertia=0.2, w_association_emb=0.5, alpha_fixed_emb=0.95, aw_param=0.5, embedding_off=False, cmc_off=True, aw_off=False, Q_xy_scaling=0.01, Q_s_scaling=0.0001
INFO     | BoxMOT v16.0.10 🚀 Python-3.12.12 torch-2.9.0+cu126
CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
INFO     | osnet_x0_25_msmt17.pt
INFO     | [PID 24] Found existing ReID weights at osnet_x0_25_msmt17.pt; skipping download.
SUCCESS  | Loaded pretrained weights from osnet_x0_25_msmt17.pt


   ✅ Done in 274.7s (12.5 FPS) | Frames: 3425 | Dets: 15719 | IDs: 220 | Errors: 0

────────────────────────────────────────────────────────────
📹 [58/59] S05_c035 (S05/c035)
   Video: /kaggle/input/datasets/thanhnguyenle/data-aicity-2023-track-2/validation/S05/c035/vdo.avi
   ROI:   Yes
   ID offset: 4922
    Frame    200/3472 | Dets:   2 | Active:   2 | IDs:    8
    Frame    400/3472 | Dets:   2 | Active:   2 | IDs:   14
    Frame    600/3472 | Dets:   4 | Active:   3 | IDs:   24
    Frame    800/3472 | Dets:   3 | Active:   3 | IDs:   40
    Frame   1000/3472 | Dets:   3 | Active:   2 | IDs:   50
    Frame   1200/3472 | Dets:   4 | Active:   3 | IDs:   62
    Frame   1400/3472 | Dets:   7 | Active:   6 | IDs:   79
    Frame   1600/3472 | Dets:   4 | Active:   4 | IDs:   94
    Frame   1800/3472 | Dets:   2 | Active:   2 | IDs:  110
    Frame   2000/3472 | Dets:   7 | Active:   7 | IDs:  127
    Frame   2200/3472 | Dets:   4 | Active:   3 | IDs:  138
    Frame   2400/3472 | Dets:   

SUCCESS  | DeepOcSort: det_thresh=0.3, max_age=30, max_obs=50, min_hits=3, iou_threshold=0.3, per_class=True, asso_func=iou, delta_t=3, inertia=0.2, w_association_emb=0.5, alpha_fixed_emb=0.95, aw_param=0.5, embedding_off=False, cmc_off=True, aw_off=False, Q_xy_scaling=0.01, Q_s_scaling=0.0001
INFO     | BoxMOT v16.0.10 🚀 Python-3.12.12 torch-2.9.0+cu126
CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
INFO     | osnet_x0_25_msmt17.pt
INFO     | [PID 24] Found existing ReID weights at osnet_x0_25_msmt17.pt; skipping download.
SUCCESS  | Loaded pretrained weights from osnet_x0_25_msmt17.pt


   ✅ Done in 266.7s (13.0 FPS) | Frames: 3472 | Dets: 14471 | IDs: 233 | Errors: 0

────────────────────────────────────────────────────────────
📹 [59/59] S05_c036 (S05/c036)
   Video: /kaggle/input/datasets/thanhnguyenle/data-aicity-2023-track-2/validation/S05/c036/vdo.avi
   ROI:   Yes
   ID offset: 5288
    Frame    200/3432 | Dets:   1 | Active:   1 | IDs:    5
    Frame    400/3432 | Dets:   2 | Active:   2 | IDs:   11
    Frame    600/3432 | Dets:   3 | Active:   3 | IDs:   21
    Frame    800/3432 | Dets:   4 | Active:   3 | IDs:   30
    Frame   1000/3432 | Dets:   7 | Active:   6 | IDs:   41
    Frame   1200/3432 | Dets:   3 | Active:   3 | IDs:   44
    Frame   1400/3432 | Dets:   5 | Active:   4 | IDs:   54
    Frame   1600/3432 | Dets:   4 | Active:   4 | IDs:   62
    Frame   1800/3432 | Dets:   2 | Active:   2 | IDs:   71
    Frame   2000/3432 | Dets:   4 | Active:   4 | IDs:   83
    Frame   2200/3432 | Dets:   4 | Active:   3 | IDs:   92
    Frame   2400/3432 | Dets:   

In [5]:
# ══════════════════════════════════════════════════════════════════
# CELL 5: SAVE CROPS FOR STAGE 2
# ══════════════════════════════════════════════════════════════════
import random

print("=" * 70)
print("💾 SAVING CROPS FOR STAGE 2")
print("=" * 70)

# ── Split assignment ────────────────────────────────────────────
# Assign each TRACK (not frame) to a split for consistency
random.seed(42)
SPLIT_RATIOS = {"train": 0.8, "val": 0.15, "test": 0.05}

def assign_split():
    r = random.random()
    if r < SPLIT_RATIOS["train"]:   return "train"
    elif r < SPLIT_RATIOS["train"] + SPLIT_RATIOS["val"]: return "val"
    else: return "test"

track_splits = {}
for gid in all_tracklets:
    track_splits[gid] = assign_split()

# ── Create directories ──────────────────────────────────────────
for split in ["train", "val", "test"]:
    for cls_name in CLASS_NAMES.values():
        (CROPS_ROOT / split / cls_name).mkdir(parents=True, exist_ok=True)
    (CROPS_ROOT / split / "unknown").mkdir(parents=True, exist_ok=True)

# ── Group crops by video for efficient extraction ───────────────
# {vid_key: {frame_idx: [(global_id, bbox, cls, split), ...]}}
video_frame_crops = defaultdict(lambda: defaultdict(list))

for gid, entries in all_tracklets.items():
    meta = all_tracklet_meta[gid]
    vid_key = meta['vid_key']
    split = track_splits[gid]
    
    for entry in entries:
        video_frame_crops[vid_key][entry['frame']].append(
            (gid, entry['bbox'], entry['class'], split)
        )

total_crops_needed = sum(
    sum(len(crops) for crops in frames.values())
    for frames in video_frame_crops.values()
)
print(f"\n📊 Crops to extract: {total_crops_needed} across {len(video_frame_crops)} videos")

# ── Extract crops video by video ────────────────────────────────
crop_count = 0
crop_split_counts = defaultdict(int)

for vid_key in tqdm(sorted(video_frame_crops.keys()), desc="Extracting crops"):
    vid_info = video_registry[vid_key]
    frame_crops = video_frame_crops[vid_key]
    sorted_frames = sorted(frame_crops.keys())
    
    if not sorted_frames:
        continue
    
    cap = cv2.VideoCapture(vid_info['video'])
    if not cap.isOpened():
        print(f"  ❌ Could not reopen: {vid_key}")
        continue
    
    # Load ROI mask
    roi_mask = load_roi_mask(vid_info['roi'])
    
    current_frame = 0
    frame_set = set(sorted_frames)
    max_frame = max(sorted_frames)
    
    while current_frame <= max_frame:
        ret, frame = cap.read()
        if not ret:
            break
        
        if current_frame in frame_set:
            # Apply ROI if available
            if roi_mask is not None:
                frame_proc = cv2.bitwise_and(frame, frame, mask=roi_mask)
            else:
                frame_proc = frame
            
            h, w = frame_proc.shape[:2]
            
            for gid, bbox, cls, split in frame_crops[current_frame]:
                x1, y1, x2, y2 = map(int, bbox)
                
                # Clamp to frame boundaries
                x1 = max(0, min(x1, w - 1))
                y1 = max(0, min(y1, h - 1))
                x2 = max(0, min(x2, w - 1))
                y2 = max(0, min(y2, h - 1))
                
                if x2 <= x1 or y2 <= y1:
                    continue
                
                crop_img = frame_proc[y1:y2, x1:x2]
                if crop_img.size == 0:
                    continue
                
                cls_name = CLASS_NAMES.get(cls, "unknown")
                crop_filename = f"{vid_key}_track_{gid:05d}_frame_{current_frame:06d}.jpg"
                crop_path = CROPS_ROOT / split / cls_name / crop_filename
                
                cv2.imwrite(str(crop_path), crop_img)
                crop_count += 1
                crop_split_counts[split] += 1
        
        current_frame += 1
    
    cap.release()

# ── Summary ─────────────────────────────────────────────────────
print(f"\n✅ Saved {crop_count} crops total")
for split in ["train", "val", "test"]:
    print(f"  📁 {split}: {crop_split_counts.get(split, 0)} crops")
    for cls_name in list(CLASS_NAMES.values()) + ["unknown"]:
        cls_dir = CROPS_ROOT / split / cls_name
        if cls_dir.exists():
            n = len(list(cls_dir.glob("*.jpg")))
            if n > 0:
                print(f"      • {cls_name}: {n}")

print("=" * 70)

💾 SAVING CROPS FOR STAGE 2

📊 Crops to extract: 232845 across 59 videos


Extracting crops:   0%|          | 0/59 [00:00<?, ?it/s]


✅ Saved 232845 crops total
  📁 train: 188005 crops
      • vehicle: 138321
      • unknown: 49684
  📁 val: 31974 crops
      • vehicle: 23943
      • unknown: 8031
  📁 test: 12866 crops
      • vehicle: 9342
      • unknown: 3524


In [6]:
# ══════════════════════════════════════════════════════════════════
# CELL 6: SAVE TRACKLETS.JSON FOR STAGE 2  (+ bboxes for Stage 5)
# ══════════════════════════════════════════════════════════════════

print("=" * 70)
print("💾 SAVING TRACKLETS.JSON")
print("=" * 70)

MAX_BBOX_ENTRIES = 2000   # cap per track to keep file manageable

tracklets_output = {}

for gid, entries in all_tracklets.items():
    if not entries:
        continue

    meta = all_tracklet_meta[gid]
    cls = entries[0]['class']
    cls_name = CLASS_NAMES.get(cls, "unknown")
    split = track_splits[gid]

    # Build frame paths (relative to crops root)
    frame_paths = []
    for entry in entries:
        crop_filename = (f"{meta['vid_key']}_track_{gid:05d}"
                         f"_frame_{entry['frame']:06d}.jpg")
        frame_paths.append(f"{split}/{cls_name}/{crop_filename}")

    # ── bbox list for Stage 5 evaluation (MOT format) ──────────
    bbox_list = []
    step = max(1, len(entries) // MAX_BBOX_ENTRIES)
    for entry in entries[::step]:
        x1, y1, x2, y2 = entry['bbox']
        bbox_list.append({
            "frame": int(entry['frame']),
            "x1":   round(float(x1), 1),
            "y1":   round(float(y1), 1),
            "w":    round(float(x2 - x1), 1),
            "h":    round(float(y2 - y1), 1),
            "conf": round(float(entry['conf']), 4),
        })

    confidences = [e['conf'] for e in entries]
    avg_conf = sum(confidences) / len(confidences) if confidences else 0.0

    tracklets_output[f"track_{gid:05d}"] = {
        "frames": frame_paths,
        "bboxes": bbox_list,               # ← Stage 5 evaluation data
        "split": split,
        "camera_id": meta['camera'],
        "scenario": meta['scenario'],
        "vid_key": meta['vid_key'],
        "class": cls_name,
        "class_id": cls,
        "track_length": len(entries),
        "avg_confidence": round(float(avg_conf), 4),
        "start_frame": entries[0]['frame'],
        "end_frame":   entries[-1]['frame'],
        "start_time":  round(float(entries[0]['timestamp']), 4),
        "end_time":    round(float(entries[-1]['timestamp']), 4),
        "duration":    round(entries[-1]['timestamp'] - entries[0]['timestamp'], 2),
    }

# Save
with open(TRACKLETS_PATH, 'w') as f:
    json.dump(tracklets_output, f, indent=2)

size_mb = TRACKLETS_PATH.stat().st_size / (1024 * 1024)
total_bboxes = sum(len(t['bboxes']) for t in tracklets_output.values())

print(f"\n✅ Saved: {TRACKLETS_PATH}")
print(f"  • Size          : {size_mb:.1f} MB")
print(f"  • Tracklets     : {len(tracklets_output)}")
print(f"  • BBox entries  : {total_bboxes:,}  (cap {MAX_BBOX_ENTRIES}/track)")

split_counts = defaultdict(int)
for t in tracklets_output.values():
    split_counts[t['split']] += 1
print(f"\n📊 TRACKLETS BY SPLIT:")
for sp in ["train", "val", "test"]:
    print(f"  {sp}: {split_counts.get(sp, 0)}")

sc_counts = defaultdict(int)
for t in tracklets_output.values():
    sc_counts[t['scenario']] += 1
print(f"\n📊 TRACKLETS BY SCENARIO:")
for sc, c in sorted(sc_counts.items()):
    print(f"  {sc}: {c}")

print("\n✅ tracklets.json now includes bbox data for Stage 5 evaluation.")
print("=" * 70)


💾 SAVING TRACKLETS.JSON

✅ Saved: /kaggle/working/tracklets.json
  • Size          : 47.0 MB
  • Tracklets     : 3955
  • BBox entries  : 232,845  (cap 2000/track)

📊 TRACKLETS BY SPLIT:
  train: 3159
  val: 589
  test: 207

📊 TRACKLETS BY SCENARIO:
  S01: 522
  S02: 478
  S03: 145
  S04: 471
  S05: 2339

✅ tracklets.json now includes bbox data for Stage 5 evaluation.


In [7]:
# ══════════════════════════════════════════════════════════════════
# CELL 7: SAVE PER-VIDEO STATS & FINAL DISK CHECK
# ══════════════════════════════════════════════════════════════════

# Save per-video stats
stats_path = OUTPUT_ROOT / "video_stats.json"
with open(stats_path, 'w') as f:
    json.dump(all_stats, f, indent=2)
print(f"✅ Video stats saved: {stats_path}")

# Final disk check
print(f"\n💾 FINAL DISK USAGE:")
for path in ["/kaggle/working"]:
    usage = shutil.disk_usage(path)
    print(f"  {path}: {usage.used/1e9:.1f}GB used / {usage.free/1e9:.1f}GB free")

# List output files
print(f"\n📁 OUTPUT FILES in /kaggle/working/:")
for f in sorted(OUTPUT_ROOT.glob("*")):
    if f.is_file():
        print(f"  📄 {f.name} ({f.stat().st_size/1e6:.1f} MB)")
    elif f.is_dir():
        n_files = sum(1 for _ in f.rglob("*.jpg"))
        print(f"  📂 {f.name}/ ({n_files} images)")

print("\n🎉 Pipeline complete! Ready for Stage 2.")

✅ Video stats saved: /kaggle/working/video_stats.json

💾 FINAL DISK USAGE:
  /kaggle/working: 6.0GB used / 14.9GB free

📁 OUTPUT FILES in /kaggle/working/:
  📄 __notebook__.ipynb (0.3 MB)
  📂 crops/ (232845 images)
  📄 osnet_x0_25_msmt17.pt (3.1 MB)
  📄 tracklets.json (49.3 MB)
  📄 video_stats.json (0.0 MB)

🎉 Pipeline complete! Ready for Stage 2.


In [8]:
# ══════════════════════════════════════════════════════════════════
# CELL 8 (OPTIONAL): ZIP OUTPUT FOR DOWNLOAD
# ══════════════════════════════════════════════════════════════════
import subprocess

print("🗜️ Zipping output...")

zip_path = OUTPUT_ROOT / "tracking_output.zip"

subprocess.run([
    "zip", "-r", "-q",
    str(zip_path),
    "crops/",
    "tracklets.json",
    "video_stats.json",
], cwd=str(OUTPUT_ROOT))

if zip_path.exists():
    size_gb = zip_path.stat().st_size / 1e9
    print(f"✅ Zip created: {zip_path} ({size_gb:.2f} GB)")
else:
    print("❌ Zip creation failed.")

# Final disk check
usage = shutil.disk_usage("/kaggle/working")
print(f"💾 /kaggle/working: {usage.used/1e9:.1f}GB used / {usage.free/1e9:.1f}GB free")

🗜️ Zipping output...
✅ Zip created: /kaggle/working/tracking_output.zip (5.47 GB)
💾 /kaggle/working: 11.5GB used / 9.4GB free
